In [2]:
from dataset import InferenceImageDataset

from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # values from efficientnet_b3
                         std=[0.229, 0.224, 0.225]), # values from efficientnet_b3
])

inference_dataset = InferenceImageDataset(
    img_dir="C:/git/RailwAI/images/inference",
    transform=transform
)

inference_dataloader = DataLoader(
    inference_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Inference images:", len(inference_dataset))

Inference images: 75


#### Load the trained model

In [3]:
import torch
import torch.nn as nn

from dataset import CustomImageDataset

from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"

# pretrained weights
weights = EfficientNet_B3_Weights.IMAGENET1K_V1
model = efficientnet_b3(weights=weights)

num_features = model.classifier[1].in_features  # 2048 for B5
model.classifier[1] = nn.Linear(num_features, 2)  # 3 = number of classes

# fine-tuned checkpoint
model.load_state_dict(torch.load("efficientnet_b3_traintyp.pth", map_location=device))
model.to(device)
model.eval()

############################################################

model.load_state_dict(
    torch.load("efficientnet_b3_traintyp.pth", map_location=device)
)

model.to(device)
model.eval()

train_dir = 'C:/git/RailwAI/images/AI-train/train'
train_dataset = CustomImageDataset(img_dir=train_dir, transform=transform)

classes = list(train_dataset.class_to_idx.keys())

## Run the inference (save as a csv)

In [6]:
import pandas as pd
import torch
import os

results = []

with torch.no_grad():
    for imgs, paths in inference_dataloader:

        imgs = imgs.to(device)

        outputs = model(imgs)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        for path, pred, prob in zip(paths, preds, probs):

            pred_class = classes[pred.item()]
            confidence = prob[pred.item()].item() * 100

            print(f"{path} -> {pred_class}: {confidence:.2f}%")

            results.append({
                "image": path,
                "prediction": pred_class,
                "confidence": confidence
            })


df_results = pd.DataFrame(results)

os.makedirs("model", exist_ok=True)

df_results.to_csv(
    "model/inference_results.csv",
    index=False
)

df_results.head()

C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0035.png -> Trieb: 99.40%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0113.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0133.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0151.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0230.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0332.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0351.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0431.png -> Trieb: 99.98%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0536.png -> Trieb: 99.78%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0606.png -> Trieb: 99.59%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0635.png -> Trieb: 99.69%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0735.png -> Trieb: 99.40%
C:/git/RailwAI/images/inference\D_

,image,prediction,confidence
0,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,99.401420
1,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,99.998212
2,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,99.997365
3,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,99.999619
4,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,99.995053
